<a href="https://colab.research.google.com/github/romavallejo/TC3009C.600_AIClass/blob/main/P1_M1_%20PythonLib_Statistics/ecobici20260813.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import io

In [2]:
from requests.exceptions import Timeout
# FASE 1: EXTRACIÓN
url = "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv"

csv_file_name = "2026-07.csv"
try:
  response = requests.get(url, timeout=1200)
  response.raise_for_status()
  print("exito")
except requests.exceptions.Timeout as e:
  print(f"Error de tiempo de espera: {e}")
  df_raw = pd.DataFrame()
except requests.exceptions.RequestException as e:
  print(f"Error durante descarga : {e}")
  df_raw = pd-pd.DataFrame()

exito


In [3]:
with open(csv_file_name, 'wb') as f:
  f.write(response.content)
print(f"Archivo CSV guardado como {csv_file_name}")

print(f"Leyendo datos desde: {csv_file_name}")
df_raw = pd.read_csv(csv_file_name)
print(f"Núm de registros: {df_raw.shape[0]}")

Archivo CSV guardado como 2026-07.csv
Leyendo datos desde: 2026-07.csv
Núm de registros: 1493484


In [4]:
print(f"Tamaño del DataFrame: {df_raw.shape}")
print(f"Previsualización del DataFrame:")
display(df_raw.head())

Tamaño del DataFrame: (1493484, 9)
Previsualización del DataFrame:


,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo
0,F,26.0,5552989,085,30/06/2026,23:43:41,503,01/07/2026,00:00:00
1,M,33.0,5128335,259,30/06/2026,23:52:28,266-267,01/07/2026,00:00:00
2,M,34.0,8647703,040,30/06/2026,23:41:57,011,01/07/2026,00:00:03
3,M,34.0,5633250,492,30/06/2026,23:56:52,489,01/07/2026,00:00:03
4,O,41.0,8516015,133,30/06/2026,23:31:01,345,01/07/2026,00:00:05


In [5]:
# FASE 2: TRANSOFMRACIÓN
df = df_raw.copy()

df['Fecha_Retiro'] = pd.to_datetime(df['Fecha_Retiro'], dayfirst=True)
df['Fecha_Arribo'] = pd.to_datetime(df['Fecha_Arribo'], dayfirst=True)
print("Columnas de fecha convertidas a datetime.")
display(df.head())

Columnas de fecha convertidas a datetime.


,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo
0,F,26.0,5552989,085,2026-06-30,23:43:41,503,2026-07-01,00:00:00
1,M,33.0,5128335,259,2026-06-30,23:52:28,266-267,2026-07-01,00:00:00
2,M,34.0,8647703,040,2026-06-30,23:41:57,011,2026-07-01,00:00:03
3,M,34.0,5633250,492,2026-06-30,23:56:52,489,2026-07-01,00:00:03
4,O,41.0,8516015,133,2026-06-30,23:31:01,345,2026-07-01,00:00:05


In [10]:
# Feature Engineering

#para calcular la duracion real combinamos fecha con la hora especifica
df['Fecha_Retiro_Completa'] = pd.to_datetime(df['Fecha_Retiro'].dt.strftime('%Y-%m-%d') + ' ' + df['Hora_Retiro'])
df['Fecha_Arribo_Completa'] = pd.to_datetime(df['Fecha_Arribo'].dt.strftime('%Y-%m-%d') + ' ' + df['Hora_Arribo'])

# duracion del viaje en minutos
df['duracion_minutos'] = (df['Fecha_Arribo_Completa'] - df['Fecha_Retiro_Completa']).dt.total_seconds() / 60
#dia de la semana
df['dia_semana'] = df['Fecha_Retiro'].dt.dayofweek
# hora del día
df['hora_dia'] = df['Fecha_Retiro'].dt.hour
# categoria de dia
df['tipo_dia'] = df['dia_semana'].apply(lambda x: 'Entre Semana' if x < 5 else 'fin de semana')

print("Columnas de fecha transformadas y calculadas.")
display(df.head())

Columnas de fecha transformadas y calculadas.


,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo,duracion_minutos,dia_semana,hora_dia,tipo_dia,Fecha_Retiro_Completa,Fecha_Arribo_Completa
0,F,26.0,5552989,085,2026-06-30,23:43:41,503,2026-07-01,00:00:00,16.316667,1,0,Entre Semana,2026-06-30 23:43:41,2026-07-01 00:00:00
1,M,33.0,5128335,259,2026-06-30,23:52:28,266-267,2026-07-01,00:00:00,7.533333,1,0,Entre Semana,2026-06-30 23:52:28,2026-07-01 00:00:00
2,M,34.0,8647703,040,2026-06-30,23:41:57,011,2026-07-01,00:00:03,18.100000,1,0,Entre Semana,2026-06-30 23:41:57,2026-07-01 00:00:03
3,M,34.0,5633250,492,2026-06-30,23:56:52,489,2026-07-01,00:00:03,3.183333,1,0,Entre Semana,2026-06-30 23:56:52,2026-07-01 00:00:03
4,O,41.0,8516015,133,2026-06-30,23:31:01,345,2026-07-01,00:00:05,29.066667,1,0,Entre Semana,2026-06-30 23:31:01,2026-07-01 00:00:05


In [12]:
#normaliazación / estandarización
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df['duracion_nomralizada'] = scaler.fit_transform(df[['duracion_minutos']])
#duración normalizada

# encoding de variables categóricas
# columna tipo_dia es categórica, la convertiremos a números usando one-hot encoding
df_encoded = pd.get_dummies(df, columns=['tipo_dia'], prefix='dia')
# con onehot encoding

#balanceo de clases
#imagenimoes que queremos predecir si un viaje es muy largo
df['viaje_largo'] = df['duracion_minutos'] > 60
print(df['viaje_largo'].value_counts())

df_encoded.head()

viaje_largo
False    1484173
True        9311
Name: count, dtype: int64


,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo,duracion_minutos,dia_semana,hora_dia,Fecha_Retiro_Completa,Fecha_Arribo_Completa,duracion_nomralizada,dia_Entre Semana,dia_fin de semana
0,F,26.0,5552989,085,2026-06-30,23:43:41,503,2026-07-01,00:00:00,16.316667,1,0,2026-06-30 23:43:41,2026-07-01 00:00:00,0.000008,True,False
1,M,33.0,5128335,259,2026-06-30,23:52:28,266-267,2026-07-01,00:00:00,7.533333,1,0,2026-06-30 23:52:28,2026-07-01 00:00:00,0.000004,True,False
2,M,34.0,8647703,040,2026-06-30,23:41:57,011,2026-07-01,00:00:03,18.100000,1,0,2026-06-30 23:41:57,2026-07-01 00:00:03,0.000009,True,False
3,M,34.0,5633250,492,2026-06-30,23:56:52,489,2026-07-01,00:00:03,3.183333,1,0,2026-06-30 23:56:52,2026-07-01 00:00:03,0.000001,True,False
4,O,41.0,8516015,133,2026-06-30,23:31:01,345,2026-07-01,00:00:05,29.066667,1,0,2026-06-30 23:31:01,2026-07-01 00:00:05,0.000015,True,False
